# Multirotor2: h/RD를 포함한 5채널 공동 학습
aerodownwash12@gmail.com 계정으로 Drive를 연결하세요. 기존 multirotor 노트와 결과 폴더는 별도입니다.
입력: L/D, sin(theta), cos(theta), z/R_D, h/RD. 출력: ur, utheta, uz.
CSV 필수: folder, L_over_D, h/RD, ground_z. RD는 생략 시 9.3101751 m입니다.
h/RD는 지면에서 로터면까지의 거리/RD이며 rotor_z 열을 사용하지 않습니다.
ground_z는 실제 CFD 지면 좌표입니다. 내부 rotor_z = ground_z + (h/RD)*RD.
원통 표시와 검증 통계는 지면부터 마지막 z격자점까지 전부 포함합니다.
런타임에서 GPU를 선택하고 위에서 아래로 실행하세요. CFD 경로와 CSV를 설정 셀에서 지정하세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from datetime import datetime
import os
import shutil
import subprocess
import sys
import uuid

# Google Drive의 '공유 문서함'에 있는 B(P4)를 '내 드라이브' 루트에
# 바로가기로 추가한 구성을 사용합니다. 바로가기 이름도 B(P4)로 유지하세요.
DRIVE_DATA = Path('/content/drive/MyDrive/B(P4)')
DRIVE_CSV = DRIVE_DATA / 'cases.csv'
DRIVE_RESULTS = Path('/content/drive/MyDrive/multirotor2_results/B(P4)')
DRIVE_DATASET = DRIVE_RESULTS / 'dataset_zrd'
DRIVE_DATASET_SA = DRIVE_DATASET.with_name('dataset_zrd_sa')
REBUILD_DATASET = False  # 기존 FATO/SA 파일은 보존하며 누락된 영역만 추출
EPOCHS = 100
BATCH_SIZE = 4
LAMBDA_GRADIENT = float(input("lambda_gradient (fixed during this training run): "))

REPO = Path('/content/multirotor2')
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]
WORK = Path('/content/multirotor2_runs') / RUN_ID
RUN_RESULTS = DRIVE_RESULTS  # 동일한 결과 폴더와 파일을 계속 사용합니다.
RAW = WORK / 'raw'
CSV = WORK / 'cases.csv'
DATASET = WORK / 'dataset_zrd'
DATASET_SA = WORK / 'dataset_zrd_sa'
MODELS = RUN_RESULTS / 'models_zrd'
MODELS_SA = RUN_RESULTS / 'models_zrd_sa'
RESULTS = RUN_RESULTS / 'results'

def run(*args):
    command = [str(x) for x in args]
    tail = []
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1,
                          env={**os.environ, 'MPLBACKEND': 'Agg',
                               'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'}) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line)
            tail = tail[-40:]
        code = process.wait()
    if code:
        details = ''.join(tail).strip() or '(process produced no output)'
        raise RuntimeError(f"Command failed (exit {code}): {command[0]}\n{details}")

REPO_BRANCH = 'main'
if not REPO.exists():
    run('git', 'clone', '--branch', REPO_BRANCH, 'https://github.com/jo-ban/multirotor2.git', REPO)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('코드에 미저장 변경이 있습니다. 보존 후 다시 실행하세요.')
    run('git', '-C', REPO, 'fetch', 'origin', REPO_BRANCH)
    run('git', '-C', REPO, 'switch', REPO_BRANCH)
    run('git', '-C', REPO, 'pull', '--ff-only', 'origin', REPO_BRANCH)
CODE = REPO / 'work_multirotor'
run(sys.executable, '-m', 'pip', 'install', '-r', CODE / 'requirements.txt')
print('이번 실행 결과:', RUN_RESULTS)


CSV와 FATO/SA 캐시를 각각 검사합니다. 기존 데이터·모델은 보존하고 누락된 영역만 생성합니다.


In [ ]:
if not DRIVE_DATA.is_dir():
    raise FileNotFoundError(DRIVE_DATA)
if not DRIVE_CSV.is_file():
    raise FileNotFoundError(DRIVE_CSV)
if DRIVE_RESULTS.resolve().is_relative_to(DRIVE_DATA.resolve()):
    raise ValueError('DRIVE_RESULTS는 DRIVE_DATA 바깥에 설정하세요.')
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import importlib
import data_utils_cylindrical
importlib.reload(data_utils_cylindrical)
import dataset_cache
importlib.reload(dataset_cache)
from dataset_cache import dataset_status, sync_dataset
import pandas as pd
case_table = pd.read_csv(DRIVE_CSV, dtype={'folder': str})
print('CSV:', DRIVE_CSV, '| 케이스 행:', len(case_table))
print(case_table.to_string(index=False))
ready, reason = dataset_status(DRIVE_DATASET, DRIVE_CSV)
print('캐시 확인:', reason)
from data_utils_cylindrical import CYLINDER_RADII_OVER_RD, SA_RADIUS_OVER_RD
FATO_CACHE_READY = ready
SA_CACHE_READY, sa_reason = dataset_status(DRIVE_DATASET_SA, DRIVE_CSV, SA_RADIUS_OVER_RD)
print('SA 캐시 확인:', sa_reason)
USE_CACHED_DATASET = FATO_CACHE_READY and SA_CACHE_READY and not REBUILD_DATASET
WORK.mkdir(parents=True, exist_ok=True)
if USE_CACHED_DATASET:
    print('전체 케이스가 확인된 캐시를 재사용합니다.')
else:
    print('누락된 영역을 추출합니다. 기존 FATO/SA 데이터는 보존합니다.')
    shutil.copytree(DRIVE_DATA, RAW, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns('*.gsheet', '*.gdoc', '*.gslides', '*.gdraw', '*.gform'))
shutil.copy2(DRIVE_CSV, CSV)
RUN_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy2(CSV, RUN_RESULTS / 'cases.csv')
revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
(RUN_RESULTS / 'code_revision.txt').write_text(revision)


## 1. 추출 및 무차원화
케이스 하나라도 실패하면 셀이 실패합니다. 오류를 해결하고 추출을 완료한 다음 학습하세요.

In [ ]:
if USE_CACHED_DATASET:
    sync_dataset(DRIVE_DATASET, DATASET)
    sync_dataset(DRIVE_DATASET_SA, DATASET_SA)
else:
    import tempfile
    fresh_dataset = Path(tempfile.mkdtemp(prefix='extract_', dir=WORK))
    if FATO_CACHE_READY:
        sync_dataset(DRIVE_DATASET, fresh_dataset)
    if SA_CACHE_READY:
        sync_dataset(DRIVE_DATASET_SA, DATASET_SA)
    run(sys.executable, '-u', CODE / 'extract_nd.py',
        '--csv', CSV, '--data-root', RAW, '--output-root', fresh_dataset,
        '--sa-output-root', DATASET_SA, '--radii-over-rd', *CYLINDER_RADII_OVER_RD)
    for root, radius in ((fresh_dataset, CYLINDER_RADII_OVER_RD[0]), (DATASET_SA, SA_RADIUS_OVER_RD)):
        ready, reason = dataset_status(root, CSV, radius)
        if not ready:
            raise RuntimeError(reason)
    if not FATO_CACHE_READY:
        sync_dataset(fresh_dataset, DRIVE_DATASET)
    if not SA_CACHE_READY:
        sync_dataset(DATASET_SA, DRIVE_DATASET_SA)
    sync_dataset(fresh_dataset, DATASET)
for root, radius in ((DATASET, CYLINDER_RADII_OVER_RD[0]), (DATASET_SA, SA_RADIUS_OVER_RD)):
    ready, reason = dataset_status(root, CSV, radius)
    print('학습 전 확인:', root.name, reason)
    if not ready:
        raise RuntimeError(reason)
(RUN_RESULTS / 'dataset_cache_path.txt').write_text(str(DRIVE_DATASET))


## 2. 5채널 공동 학습
FATO/SA 각각 입력 5채널 → 원통속도 출력 3채널로 학습합니다. 기존 4채널 가중치는 사용할 수 없습니다.
손실은 속도 L1 + lambda_gradient × 표면 기울기 L1이며, 지정한 epoch 수만큼 매번 학습합니다.


In [ ]:
run(sys.executable, CODE / 'train_nd.py',
    '--csv', CSV, '--data-root', DATASET, '--model-dir', MODELS,
    '--sa-data-root', DATASET_SA, '--sa-model-dir', MODELS_SA,
    '--epochs', EPOCHS, '--batch-size', BATCH_SIZE,
    '--lambda-gradient', LAMBDA_GRADIENT)


## 3. 높이 조건을 지정해 예측
PREDICT_L_OVER_D와 PREDICT_H_OVER_RD를 바꾸세요. 높이가 None이면 기준 NPZ의 h/RD를 사용합니다.
ground_z와 z_max는 기준 CFD NPZ에서 읽습니다. 모든 z격자점이 PNG/3D 원통에 표시됩니다.


In [ ]:
import numpy as np

# RD is fixed; vary L/D and the dimensionless rotor height h/RD.
FIXED_RD_M = 9.3101751
PREDICT_L_OVER_D = 1.5
PREDICT_H_OVER_RD = None  # Set dimensionless h/RD, or use reference case height
# Choose a reference input filename, or leave None to use the first extracted case.
REFERENCE_INPUT_NAME = None
input_paths = sorted((DATASET / 'inputs').glob('input_*.npz'))
if not input_paths:
    raise FileNotFoundError('Run extraction first.')
reference_path = DATASET / 'inputs' / REFERENCE_INPUT_NAME if REFERENCE_INPUT_NAME else input_paths[0]
with np.load(reference_path) as meta:
    reference_rd = float(meta['disk_radius_m'])
    REFERENCE_H_OVER_RD = float(meta['height_over_rd'])
    GROUND_Z_M = float(meta['ground_z_m'])
    Z_MAX_M = float(meta['z_max_m'])
if not np.isclose(reference_rd, FIXED_RD_M, rtol=1e-6, atol=1e-7):
    raise ValueError('Extracted RD does not match FIXED_RD_M. Check the CSV before prediction.')
print('Reference:', reference_path.name)
print('RD, L/D:', FIXED_RD_M, PREDICT_L_OVER_D)
H_OVER_RD = REFERENCE_H_OVER_RD if PREDICT_H_OVER_RD is None else float(PREDICT_H_OVER_RD)
print('h/RD, ground_z, mesh z_max:', H_OVER_RD, GROUND_Z_M, Z_MAX_M)
run(sys.executable, CODE / 'predict_nd.py',
    '--model-dir', MODELS, '--sa-model-dir', MODELS_SA, '--output', RESULTS / 'prediction.npz',
    '--rd', FIXED_RD_M, '--l-over-d', PREDICT_L_OVER_D,
    '--h-over-rd', H_OVER_RD, '--ground-z', GROUND_Z_M,
    '--z-max', Z_MAX_M, '--no-show', '--plot-output', RESULTS / 'prediction.png')
from IPython.display import display, Image
display(Image(filename=str(RESULTS / 'prediction.png')))
import importlib.util
spec = importlib.util.spec_from_file_location('prediction_viewer', CODE / 'visualize_prediction_3d.py')
viewer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(viewer)
figure = viewer.build_figure(RESULTS / 'prediction.npz')
figure.show(renderer='colab', auto_play=False)
print('Interactive HTML:', RESULTS / 'prediction_3d.html')

# Download is optional: both HTML and PNG are already saved to Drive.
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from colab_results import download_button
download_button(RESULTS / 'prediction_3d.html')


## 4. 검증 케이스 오차 평가 (선택)
CSV의 `type=V` 케이스만 평가합니다. 학습에서 제외된 실제 추출값과 동일 좌표의 모델 예측값을 비교합니다.
HTML은 FATO / SA 영역 선택 아래 성분 버튼으로 U_r, U_theta, U_z, 속력을 선택하고 CFD / 예측 / 절대오차 3패널을 표시합니다. SA가 없으면 SA 데이터 없음으로 안내합니다. PNG는 기존 4행 × 3열을 유지합니다.
HTML·비교 NPZ·PNG·오차 CSV는 Drive에 저장됩니다. 아래에서 케이스 선택, 3D 회전/2D 전개면 전환, HTML 다운로드가 가능합니다.


In [ ]:
import pandas as pd
from IPython.display import display
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from colab_results import show_validation_results

table = pd.read_csv(CSV)
if 'type' in table and table['type'].astype(str).str.strip().str.upper().eq('V').any():
    run(sys.executable, CODE / 'evaluate_error.py',
        '--csv', CSV, '--data-root', DATASET, '--model-dir', MODELS,
        '--sa-data-root', DATASET_SA, '--sa-model-dir', MODELS_SA,
        '--output-dir', RESULTS / 'evaluation', '--no-show')
    metrics = pd.read_csv(RESULTS / 'evaluation' / 'validation_errors_cyl2rd.csv')
    display(metrics)
    show_validation_results(RESULTS / 'evaluation', metrics['case_id'].tolist())
else:
    print('type=V 케이스가 없어 평가를 건너뜁니다.')
print('Drive 저장 위치:', RUN_RESULTS)


## RD 범위 그래프와 최대 오차
RD는 기존 외접 반경[m]이며 z는 기존 절대 높이[m]입니다. 그래프 상단을 z=RD로 제한하고 기존 하단은 유지합니다.
U_r, U_theta, U_z, 속력별로 표시 구간의 전체 원본 격자에서 A(최대 절대오차, m/s), B(최대 위치별 오차율, %)를 찾습니다.
주석은 서로 분리하며 theta[deg], x/y/z[m]와 값을 표시합니다. x/y는 원통 중심 기준입니다.
기존 NMAE=MAE/mean(|CFD|)×100을 유지합니다. 위치별 오차율은 |예측−CFD|/|CFD|×100이며 CFD=0은 제외하고 개수를 표시합니다. 전부 0이면 N/A입니다.
동률은 z, theta 순서의 첫 지점으로 표시합니다. 3D 축소 표시와 이음새 복제점은 최대값 계산에 영향을 주지 않습니다.
아래 재생성 셀만 실행하면 기존 comparison_*.npz에서 HTML·PNG·오차 CSV를 갱신합니다. 추출·학습·모델 예측은 실행하지 않습니다.


In [ ]:
# RD 범위 HTML·PNG·오차 CSV 재생성: 이 셀만 실행 가능 (재추출·재학습·모델 예측 없음)
from pathlib import Path
import importlib.util
import subprocess
import sys

# 기존 평가 결과 comparison_*.npz가 저장된 폴더
HTML_EVALUATION_DIR = Path(globals().get(
    'RESULTS', '/content/drive/MyDrive/multirotor2_results/B(P4)/results'
)) / 'evaluation'

if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')

if not list(HTML_EVALUATION_DIR.glob('comparison_*.npz')):
    raise FileNotFoundError(
        f'기존 comparison_*.npz가 없습니다: {HTML_EVALUATION_DIR}\n'
        'HTML_EVALUATION_DIR을 기존 평가 결과 폴더로 지정하세요. 재추출·재학습은 실행하지 않습니다.'
    )

HTML_REPO = Path('/content/multirotor2')
HTML_BRANCH = 'claude/great-faraday-ijzn3f'
if HTML_REPO.exists():
    if subprocess.check_output(['git', '-C', str(HTML_REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('코드 변경을 먼저 보존하세요.')
    subprocess.run(['git', '-C', str(HTML_REPO), 'fetch', 'origin', HTML_BRANCH], check=True)
    subprocess.run(['git', '-C', str(HTML_REPO), 'switch', HTML_BRANCH], check=True)
    subprocess.run(['git', '-C', str(HTML_REPO), 'pull', '--ff-only', 'origin', HTML_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', HTML_BRANCH, 'https://github.com/jo-ban/multirotor2.git',
                    str(HTML_REPO)], check=True)

missing = [name for name in ('numpy', 'plotly', 'matplotlib') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)

subprocess.run([
    sys.executable, str(HTML_REPO / 'work_multirotor' / 'evaluate_error.py'),
    '--replot', '--output-dir', str(HTML_EVALUATION_DIR)
], check=True)
print('HTML 저장 위치:', HTML_EVALUATION_DIR)

# Refresh the in-notebook viewer too, including an already-running Colab session.
code_dir = str(HTML_REPO / 'work_multirotor')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
import importlib
for module_name in ('data_utils_cylindrical', 'display_analysis', 'visualize_validation', 'colab_results'):
    module = importlib.import_module(module_name)
    importlib.reload(module)
from colab_results import show_validation_results
from IPython.display import display
import pandas as pd
display(pd.read_csv(HTML_EVALUATION_DIR / 'validation_errors_cyl2rd.csv'))
case_ids = [p.stem.removeprefix('comparison_') for p in sorted(HTML_EVALUATION_DIR.glob('comparison_*.npz'))]
show_validation_results(HTML_EVALUATION_DIR, case_ids)
